In [1]:
import numpy as np
import pandas as pd
from scipy.special import lpmv
import pathlib
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import json
import xarray as xr
from stl import mesh
import scipy as sp
import harmonica as hm
import verde as vd
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


import sys
sys.path.append("..")

import multipoles as mt
import micromag as mg
from micromag import MICROMETER_TO_METER, TESLA_TO_NANOTESLA

In [2]:
# Directories
grid_dir = pathlib.Path("../../data/micromag_models/tecfiles/grids")
moments_csv = pathlib.Path("../../data/micromag_models/real_micromagnetic_moments/dipole_moments.csv")
stl_dir = pathlib.Path("../..") / "data/micromag_models/STL_ellipsoids"

output_images = pathlib.Path("../../data/noise_levels/images")
output_results = pathlib.Path("../../data/noise_levels")
output_images.mkdir(parents=True, exist_ok=True)  
output_results.mkdir(parents=True, exist_ok=True) 

In [3]:
def angular_distance(vec_a, vec_b):
    # unit vector
    vec_a = vec_a / np.linalg.norm(vec_a)
    vec_b = vec_b / np.linalg.norm(vec_b)

    cos_theta = np.dot(vec_a, vec_b)
    cos_theta = np.clip(cos_theta, -1.0, 1.0)

    return np.rad2deg(np.arccos(cos_theta))

def goal_function_dipole(parameter, obs_data, coordinates, amplitude, x_0, y_0, z_0):
    bz = mg.dipole_bz(coordinates, 
                      ([parameter[0]*x_0], [parameter[1]*y_0], [parameter[2]*z_0]),
                     np.array([parameter[3], parameter[4], parameter[5]])*amplitude
                     )
    
    Res = np.linalg.norm(obs_data-bz.ravel())
    return(Res)

In [9]:
# Read real magnetic moments from CSV file
moments_df = pd.read_csv(moments_csv).set_index("name")

# Llist of .nc files
grid_files = sorted(grid_dir.glob("*.nc"))
selected_eps = {"eps2", "eps7", "eps12", "eps17"} # intermediate of each shape

filtered_files = [
    f for f in grid_files
    if f.stem.split("_")[1] in selected_eps
]

filtered_files



[PosixPath('../../data/micromag_models/tecfiles/grids/grid_eps12_0.05.nc'),
 PosixPath('../../data/micromag_models/tecfiles/grids/grid_eps12_0.075.nc'),
 PosixPath('../../data/micromag_models/tecfiles/grids/grid_eps12_0.1.nc'),
 PosixPath('../../data/micromag_models/tecfiles/grids/grid_eps12_0.15.nc'),
 PosixPath('../../data/micromag_models/tecfiles/grids/grid_eps12_0.2.nc'),
 PosixPath('../../data/micromag_models/tecfiles/grids/grid_eps12_0.25.nc'),
 PosixPath('../../data/micromag_models/tecfiles/grids/grid_eps12_0.35.nc'),
 PosixPath('../../data/micromag_models/tecfiles/grids/grid_eps12_0.5.nc'),
 PosixPath('../../data/micromag_models/tecfiles/grids/grid_eps12_0.75.nc'),
 PosixPath('../../data/micromag_models/tecfiles/grids/grid_eps12_1.0.nc'),
 PosixPath('../../data/micromag_models/tecfiles/grids/grid_eps12_1.25.nc'),
 PosixPath('../../data/micromag_models/tecfiles/grids/grid_eps12_1.5.nc'),
 PosixPath('../../data/micromag_models/tecfiles/grids/grid_eps12_10.0.nc'),
 PosixPath('../.

In [5]:
noise_levels = [20, 100]  # nT

In [6]:
results = []

for file in filtered_files:
    model_name = file.stem.split("_")[1]
    height = float(file.stem.split("_")[2].replace("um", ""))

    for noise in noise_levels:
        print(f"Processing {model_name} at {height} µm with noise {noise} nT")

        model_name = file.stem.split("_")[1]  
        height = float(file.stem.split("_")[2].replace("um", ""))
        
        # Read the grid
        data = xr.load_dataset(file)
        
        dipole_moments = data.magnetic_moment.values
        dipole_coordinates = data.centroid.values
        # Upward continuation (small to avoid border effects)
        height_difference = 1.0e-60
        data = (
            hm.upward_continuation(data.bz, height_difference)
            .assign_attrs(data.bz.attrs)
            .to_dataset(name="bz")
            .assign_coords(x=data.x, y=data.y)
            .assign_coords(z=data.z + height_difference)
        )
        data = data.assign(mg.data_gradients(data.bz))
        
        #############################################################
        # Add Gaussian noise to bz
        noise_array = np.random.normal(0, noise, size=data.bz.shape)
        data["bz"] = data.bz + noise_array
        #############################################################
        
        coordinates = np.meshgrid(data.x.values, data.y.values, indexing="xy")
        coordinates = [
            coordinates[0],
            coordinates[1],
            np.full_like(coordinates[0], data.z.values.ravel()[0])
        ]
    
        ### DIPOLAR LINEAR ###
        ##################################################################################
        position, base_level = mg.euler_deconvolution(
            data.bz, data.x_deriv, data.y_deriv, data.z_deriv
        )
        data_minus_background = data.bz.values - base_level
        data = data.assign(data_minus_background=(['y','x'], data_minus_background))
        moment_dip_lin, *_ = mg.dipole_moment_inversion(data.data_minus_background, position)
        angular_dip_lin = angular_distance(dipole_moments, moment_dip_lin)
        _, _, amp_real = mg.vector_to_angles(dipole_moments)
        _, _, amp_model = mg.vector_to_angles(moment_dip_lin)
        moment_dip_misfit_lin = abs((amp_real - amp_model) / amp_real) * 100
        pos_dip_misfit_lin = np.linalg.norm(np.array(dipole_coordinates) - np.array(position))
    
        
        # # plot results for linear dipolar model
        pred_dip_lin = mg.dipole_bz([coordinates[0], coordinates[1], data.z], position, moment_dip_lin)    
        data = data.assign(pred_dip_lin=(['y','x'],pred_dip_lin))
        data.pred_dip_lin.attrs = {"long_name": "predicted-data", "units": "nT"}
        residual_dip_lin = data.bz.values - data.pred_dip_lin.values
        data = data.assign(residual_dip_lin=(['y','x'],residual_dip_lin))
        data.residual_dip_lin.attrs = {"long_name": "residual-data", "units": "nT"}
           
        
        ### DIPOLAR NONLINEAR ###
        ##################################################################################
        table = vd.grid_to_table(data)
        amp = np.linalg.norm(moment_dip_lin)
        args = (table.bz.values.ravel(), (table.x.values, table.y.values, table.z.values.ravel()), amp, *position)
        opt = sp.optimize.minimize(goal_function_dipole, (1, 1, 1, *(moment_dip_lin / amp)), args=args, method="Nelder-Mead")
        if opt.success:
            moment_dip_nonlin = np.array(opt.x[3:]) * amp
            pos_dip_nonlin = np.array(opt.x[:3]) * np.array(position)
        else:
            moment_dip_nonlin = moment_dip_lin
            pos_dip_nonlin = position
        ang_dip_nonlin = angular_distance(dipole_moments, moment_dip_nonlin)
        _, _, amp_model = mg.vector_to_angles(moment_dip_nonlin)
        mom_dip_misfit_nonlin = abs((amp_real - amp_model) / amp_real) * 100
        pos_dip_misfit_nonlin = np.linalg.norm(np.array(dipole_coordinates) - pos_dip_nonlin)
    
        # # plot results for nonlinear dipolar model
        pred_dip_nonlin = mg.dipole_bz([coordinates[0], coordinates[1], data.z], pos_dip_nonlin, moment_dip_nonlin)    
        data = data.assign(pred_dip_nonlin=(['y','x'],pred_dip_nonlin))
        data.pred_dip_nonlin.attrs = {"long_name": "predicted-data", "units": "nT"}
        residual_dip_nonlin = data.bz.values - data.pred_dip_nonlin.values
        data = data.assign(residual_dip_nonlin=(['y','x'],residual_dip_nonlin))
        data.residual_dip_nonlin.attrs = {"long_name": "residual-data", "units": "nT"}
        
        ##################################################################################
        ### SPHERICAL HARMONIC ###
        
        sph = mt.SphericalHarmonicsBz(pos_dip_nonlin, min_degree=1, max_degree=10).fit(coordinates, data.data_minus_background)
        pred_shp = sph.predict(coordinates)
        moment_shp = sph.magnetic_moment_
        pos_mp_lin = sph.location
        
        ang_mp_lin = angular_distance(dipole_moments, moment_shp)
        _, _, amp_model = mg.vector_to_angles(moment_shp)
        mom_mp_misfit_lin = abs((amp_real - amp_model) / amp_real) * 100
        pos_mp_misfit_lin = np.linalg.norm(np.array(dipole_coordinates) - pos_mp_lin)
    
        # # plot results for linear multipolar model
        data = data.assign(pred_shp=(['y','x'],np.reshape(pred_shp, data.bz.shape)))
        data.pred_shp.attrs = {"long_name": "predicted-data", "units": "nT"}
        residual_mp_lin = data.bz.values - data.pred_shp.values
        data = data.assign(residual_mp_lin=(['y','x'],residual_mp_lin))
        data.residual_mp_lin.attrs = {"long_name": "residual-data", "units": "nT"}
        
    
        #########################################################################################################
        ### SPHERICAL HARMONIC WITH NONLINEAR POSITION ESTIMATION ###
        sph.update_location(coordinates, table)
        pred_shp_nl  = sph.predict(coordinates)
        moment_shp_nl = sph.magnetic_moment_
        pos_mp_nonlin = sph.location
        
        ang_mp_nonlin = angular_distance(dipole_moments, moment_shp_nl)
        _, _, model_amp = mg.vector_to_angles(moment_shp_nl)
        mom_mp_misfit_nonlin = np.abs((amp_real - model_amp) / amp_real) * 100
        pos_mp_misfit_nonlin = np.linalg.norm(np.array(dipole_coordinates) - pos_mp_nonlin)
    
        # # plot results for non-linear multipolar model
        data = data.assign(pred_shp_nl=(['y','x'],np.reshape(pred_shp_nl, data.bz.shape)))
        data.pred_shp_nl.attrs = {"long_name": "predicted-data", "units": "nT"}
        residual_mp_nonlin = data.bz.values - data.pred_shp_nl.values
        data = data.assign(residual_mp_nonlin=(['y','x'],residual_mp_nonlin))
        data.residual_mp_nonlin.attrs = {"long_name": "residual-data", "units": "nT"}
        #########################################################################################################
    
        # Append all
        results.append({
            "model": model_name,
            "height_um": height,
            "noise_nT": noise,
            "angular_dipole_linear": angular_dip_lin,
            "moment_misfit_dipole_linear": moment_dip_misfit_lin,
            "position_misfit_dipole_linear": pos_dip_misfit_lin,
            "angular_dipole_nonlinear": ang_dip_nonlin,
            "moment_misfit_dipole_nonlinear": mom_dip_misfit_nonlin,
            "position_misfit_dipole_nonlinear": pos_dip_misfit_nonlin,
            "angular_multipole_linear": ang_mp_lin,
            "moment_misfit_multipole_linear": mom_mp_misfit_lin,
            "position_misfit_multipole_linear": pos_mp_misfit_lin,
            "angular_multipole_nonlinear": ang_mp_nonlin,
            "moment_misfit_multipole_nonlinear": mom_mp_misfit_nonlin,
            "position_misfit_multipole_nonlinear": pos_mp_misfit_nonlin,
        })
    
    
        # Prepare full 4x3 panel for all inversion results
        fig, axes = plt.subplots(4, 3, figsize=(16, 20))
        cbar_kwargs = dict(orientation="horizontal", aspect=50, pad=0.1, shrink=0.9)
        
        # Campos para cada linha (modelo)
        fields_list = [
            ["bz", "pred_dip_lin", "residual_dip_lin"],
            ["bz", "pred_dip_nonlin", "residual_dip_nonlin"],
            ["bz", "pred_shp", "residual_mp_lin"],
            ["bz", "pred_shp_nl", "residual_mp_nonlin"]
        ]
        
        # Títulos por linha e coluna
        row_titles = [
            "Dipole Linear",
            "Dipole Nonlinear",
            "Spherical Harmonic Linear",
            "Spherical Harmonic Nonlinear"
        ]
        col_titles = ["Observed", "Predicted", "Residual"]
    
        position_list = [position, pos_dip_nonlin, pos_mp_lin, pos_mp_nonlin]
        
        # Plotagem com títulos
        for row_idx, fields in enumerate(fields_list):
            for col_idx, field in enumerate(fields):
                ax = axes[row_idx, col_idx]
                data[field].plot.pcolormesh(ax=ax, cmap="seismic", cbar_kwargs=cbar_kwargs)
                ax.set_aspect("equal")
                if row_idx == 0:
                    ax.set_title(col_titles[col_idx], fontsize=12, weight="bold")
                if col_idx == 0:
                    ax.set_ylabel(row_titles[row_idx], fontsize=12, weight="bold")
    
    
                if col_idx == 0:
                    # Observed → posição real
                    text = f"Real: ({dipole_coordinates[0]:.5f}, {dipole_coordinates[1]:.5f}, {dipole_coordinates[2]:.5f}) µm"
                    ax.text(
                    0.4, 0.05,
                    text,
                    transform=ax.transAxes,
                    ha="center",
                    fontsize=9
                )
                elif col_idx == 1:
                    # Predicted/Residual → posição estimada do modelo da linha
                    pos = position_list[row_idx]
                    text = f"Est: ({pos[0]:.5f}, {pos[1]:.5f}, {pos[2]:.5f}) µm"
                
                    ax.text(
                        0.4, 0.05,
                        text,
                        transform=ax.transAxes,
                        ha="center",
                        fontsize=9
                    )
        
        plt.tight_layout()
        plt.savefig(output_images / f"{model_name}_{height}um_noise{noise}.png", dpi=300)        # plt.show()
        plt.close()


Processing eps12 at 0.05 µm with noise 20 nT
Processing eps12 at 0.05 µm with noise 100 nT
Processing eps12 at 0.075 µm with noise 20 nT
Processing eps12 at 0.075 µm with noise 100 nT
Processing eps12 at 0.1 µm with noise 20 nT
Processing eps12 at 0.1 µm with noise 100 nT
Processing eps12 at 0.15 µm with noise 20 nT
Processing eps12 at 0.15 µm with noise 100 nT
Processing eps12 at 0.2 µm with noise 20 nT
Processing eps12 at 0.2 µm with noise 100 nT
Processing eps12 at 0.25 µm with noise 20 nT
Processing eps12 at 0.25 µm with noise 100 nT
Processing eps12 at 0.35 µm with noise 20 nT
Processing eps12 at 0.35 µm with noise 100 nT
Processing eps12 at 0.5 µm with noise 20 nT
Processing eps12 at 0.5 µm with noise 100 nT
Processing eps12 at 0.75 µm with noise 20 nT
Processing eps12 at 0.75 µm with noise 100 nT
Processing eps12 at 1.0 µm with noise 20 nT
Processing eps12 at 1.0 µm with noise 100 nT
Processing eps12 at 1.25 µm with noise 20 nT
Processing eps12 at 1.25 µm with noise 100 nT
Proce

In [7]:
results_df = pd.DataFrame(results)
results_df.head(-20)

,model,height_um,noise_nT,angular_dipole_linear,moment_misfit_dipole_linear,position_misfit_dipole_linear,angular_dipole_nonlinear,moment_misfit_dipole_nonlinear,position_misfit_dipole_nonlinear,angular_multipole_linear,moment_misfit_multipole_linear,position_misfit_multipole_linear,angular_multipole_nonlinear,moment_misfit_multipole_nonlinear,position_misfit_multipole_nonlinear
0,eps12,0.050,20,3.174920,181.108600,0.066764,3.114353,207.139253,0.061954,167.508503,64.471044,0.061954,0.384780,10.942192,0.036564
1,eps12,0.050,100,3.175509,181.106976,0.066765,3.175509,181.106976,0.066765,162.261210,76.845200,0.066765,0.392649,10.987349,0.036605
2,eps12,0.075,20,2.986477,174.919502,0.068496,2.986477,174.919502,0.068496,153.658747,86.321075,0.068496,0.111275,4.771605,0.065780
3,eps12,0.075,100,2.987003,174.921167,0.068496,2.846490,166.184050,0.070607,152.204582,87.869090,0.070607,0.056886,4.782614,0.065887
4,eps12,0.100,20,2.826307,143.935503,0.076220,2.640744,129.862476,0.080605,21.081045,88.314131,0.080605,0.051763,0.591278,0.083946
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,eps7,0.250,100,6.549683,28.898646,0.003459,6.547090,29.228477,0.003024,0.249504,0.488118,0.003024,0.043860,0.061829,0.164262
120,eps7,0.350,20,5.613730,16.788599,0.023915,5.598642,18.708294,0.020242,0.224257,0.486284,0.020242,0.020272,0.121070,0.205836
121,eps7,0.350,100,5.613026,16.770379,0.023943,5.597812,18.710142,0.020231,0.225470,0.505972,0.020231,0.073253,0.147991,0.270861
122,eps7,0.500,20,4.671835,6.829691,0.048456,4.641796,11.047589,0.037259,0.195070,0.275457,0.037259,0.027671,0.025540,0.302982


In [8]:
results_df.to_csv(output_results / "misfit_results_noise_levels.csv", index=False)